# ML-05 — Feature Vector and Leakage/Privacy Check

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

**Prediction moment:** the start of the last 30 days. **Label:** `stays_under` = the page's CTR in the last 30 days is below half of the pooled CTR of its narrow position bucket in that window. **Unit:** one visible page (positions 1–20, ≥ 1,000 impressions in 90 days, ≥ 100 in each 30-day window).

Every feature below is built only from the **previous 30 days** (days 31–60) plus static page context. Traffic counts are log-transformed because they are heavy-tailed; categorical fields are one-hot encoded with an explicit missing column; `word_count` gets a missing flag instead of a silent zero.

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw/content_refresh_anonymized.csv").exists())
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")

d = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 1000)
       & (df["impressions_prev_30d"] >= 100) & (df["impressions_last_30d"] >= 100)].copy()
d["bucket"] = pd.cut(d["avg_position"], [0, 3, 5, 10, 15, 20], labels=["1-3", "3-5", "5-10", "10-15", "15-20"])
for w in ["prev", "last"]:
    d[f"ctr_{w}"] = 100 * d[f"clicks_{w}_30d"] / d[f"impressions_{w}_30d"]
    t = d.groupby("bucket", observed=True)[[f"clicks_{w}_30d", f"impressions_{w}_30d"]].sum()
    d[f"exp_{w}"] = d["bucket"].map(100 * t[f"clicks_{w}_30d"] / t[f"impressions_{w}_30d"]).astype(float)
d["stays_under"] = (d["ctr_last"] < 0.5 * d["exp_last"]).astype(int)          # LABEL (later window)
d = d.reset_index(drop=True)

X = pd.DataFrame({
    "log_imp_prev": np.log1p(d["impressions_prev_30d"]),
    "log_clk_prev": np.log1p(d["clicks_prev_30d"]),
    "log_sess_prev": np.log1p(d["sessions_prev_30d"]),
    "ctr_prev": d["ctr_prev"],
    "ratio_prev": d["ctr_prev"] / d["exp_prev"],
    "avg_position": d["avg_position"],
    "content_age_days": d["content_age_days"],
    "has_word_count": d["word_count"].notna().astype(int),
    "word_count": d["word_count"].fillna(0),
})
X = pd.concat([X, pd.get_dummies(d[["main_intent", "content_type"]], dummy_na=True, dtype=int)], axis=1)
y = d["stays_under"]
print(f"Feature matrix: {X.shape[0]:,} pages x {X.shape[1]} columns, {d['client_id'].nunique()} clients | label base rate {y.mean():.1%}")
print("Any NaN left in X:", bool(X.isna().any().any()))
X.describe().T[["mean", "min", "max"]].round(2)

Feature matrix: 9,664 pages x 18 columns, 28 clients | label base rate 37.8%
Any NaN left in X: False


,mean,min,max
log_imp_prev,7.47,4.64,12.30
log_clk_prev,1.66,0.00,7.40
log_sess_prev,2.32,0.00,8.35
ctr_prev,0.32,0.00,4.31
ratio_prev,0.96,0.00,15.60
avg_position,9.01,0.20,20.00
content_age_days,259.39,90.00,557.00
has_word_count,0.68,0.00,1.00
word_count,2269.79,0.00,9157.00
main_intent_commercial,0.17,0.00,1.00


## 2. Feature notes (meaning, missing, categorical, available-when?)

Rates here are on the dataset's ×100 scale (`0.35` = 0.35%). "Available when" asks: *does this value exist before the start of the label window?*

In [2]:
notes = pd.DataFrame([
    ("log_imp_prev", "log(1 + impressions, days 31-60)", "none (>= 100 by filter)", "numeric", "yes: before label window"),
    ("log_clk_prev", "log(1 + clicks, days 31-60)", "none", "numeric", "yes"),
    ("log_sess_prev", "log(1 + GA4 sessions, days 31-60)", "none in slice", "numeric", "yes"),
    ("ctr_prev", "CTR in days 31-60", "none", "numeric", "yes"),
    ("ratio_prev", "CTR / pooled CTR of the position bucket, days 31-60", "none", "numeric", "yes (bucket CTR from the same window)"),
    ("avg_position", "average position over 90 days", "none", "numeric", "PARTLY: 90-day average overlaps the label window (known, tested in sec. 3)"),
    ("content_age_days", "days since publication", "none", "numeric", "yes: static"),
    ("has_word_count", "1 if word count is recorded", "flag itself", "binary", "yes: static"),
    ("word_count", "words on the page (0 when missing)", f"{d['word_count'].isna().mean():.0%} missing -> 0 + flag", "numeric", "yes: static"),
    ("main_intent_*", "search intent of the target keyword", f"{d['main_intent'].isna().mean():.1%} missing -> own column", "one-hot", "yes: static"),
    ("content_type_*", "article type", f"{d['content_type'].isna().mean():.1%} missing -> own column", "one-hot", "yes: static"),
], columns=["feature", "meaning", "missing handling", "type", "available before prediction?"])
print(notes.to_string(index=False))

# Missingness follows categories? (a blind fill could inject a category signal)
print("\nword_count missing rate by content type:")
print(d.groupby("content_type", dropna=False)["word_count"].apply(lambda s: s.isna().mean()).round(3).to_string())

         feature                                             meaning           missing handling    type                                               available before prediction?
    log_imp_prev                    log(1 + impressions, days 31-60)    none (>= 100 by filter) numeric                                                   yes: before label window
    log_clk_prev                         log(1 + clicks, days 31-60)                       none numeric                                                                        yes
   log_sess_prev                   log(1 + GA4 sessions, days 31-60)              none in slice numeric                                                                        yes
        ctr_prev                                   CTR in days 31-60                       none numeric                                                                        yes
      ratio_prev CTR / pooled CTR of the position bucket, days 31-60                       none numeric  

## 3. The leakage hunt

Three attacks on my own features:
1. **Single-column test.** Train a tiny model on *one* column at a time, on held-out clients. A column that alone predicts the label almost perfectly is the label in disguise.
2. **Planted leak.** Add a column I know is leaky to the full feature set. If the score doesn't jump toward 1.0, my test harness is broken.
3. **Suspect removal.** Drop `avg_position` (the one partial overlap) and see whether the honest score depends on it.

All scores: ROC-AUC, 5-fold `GroupKFold` by client, label base rate printed above.

In [3]:
from sklearn.model_selection import GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
SEED = 42
folds = list(GroupKFold(5).split(X, y, d["client_id"]))

def cv_auc(M, model):
    s = np.zeros(len(M))
    for tr, te in folds:
        s[te] = model().fit(M.iloc[tr], y.iloc[tr]).predict_proba(M.iloc[te])[:, 1]
    return roc_auc_score(y, s)

logit = lambda: make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000))
rf = lambda: RandomForestClassifier(n_estimators=200, min_samples_leaf=20, random_state=SEED, n_jobs=-1)

# 1) single-column test: my features vs. suspect columns from the dataset
suspects = pd.DataFrame({
    "ctr_last_30d (label window)": d["ctr_last"],
    "clicks_last_30d": d["clicks_last_30d"],
    "ctr (90-day, contains label window)": d["ctr"],
    "clicks_90d": np.log1p(d["clicks_90d"]),
    "trend_pct": d["trend_pct"],
    "engagement_rate (90-day)": d["engagement_rate"],
    "days_since_last_update": d["days_since_last_update"],
})
single = {f"[mine] {col}": cv_auc(X[[col]], logit) for col in ["ratio_prev", "ctr_prev", "log_clk_prev", "avg_position"]}
single.update({f"[suspect] {col}": cv_auc(suspects[[col]].fillna(0), logit) for col in suspects})
print("Single-column ROC-AUC on unseen clients (0.5 = no signal, ~1.0 = label in disguise):")
print(pd.Series(single).round(3).to_string())

# 2) + 3) planted leak and suspect removal, full model
full = {
    "honest feature set": X,
    "+ ctr_last_30d (planted leak)": X.assign(leak=d["ctr_last"]),
    "+ 90-day ctr (overlapping window)": X.assign(leak=d["ctr"]),
    "- avg_position": X.drop(columns=["avg_position"]),
}
print("\nRandom forest ROC-AUC, grouped by client:")
print(pd.Series({k: cv_auc(v, rf) for k, v in full.items()}).round(3).to_string())

# Privacy: no identifiers or free text among features
assert not any(k in col for col in X.columns for k in ["client", "content_id", "url", "title", "query"])
print("\nPrivacy check: no IDs, URLs, titles or queries in the feature matrix.")

Single-column ROC-AUC on unseen clients (0.5 = no signal, ~1.0 = label in disguise):
[mine] ratio_prev                                0.755
[mine] ctr_prev                                  0.756
[mine] log_clk_prev                              0.721
[mine] avg_position                              0.524
[suspect] ctr_last_30d (label window)            0.994
[suspect] clicks_last_30d                        0.833
[suspect] ctr (90-day, contains label window)    0.873
[suspect] clicks_90d                             0.781
[suspect] trend_pct                              0.405
[suspect] engagement_rate (90-day)               0.457
[suspect] days_since_last_update                 0.407

Random forest ROC-AUC, grouped by client:


honest feature set                   0.793
+ ctr_last_30d (planted leak)        0.996
+ 90-day ctr (overlapping window)    0.885
- avg_position                       0.791

Privacy check: no IDs, URLs, titles or queries in the feature matrix.


**What the hunt shows:**
- **The harness works.** The label-window CTR alone reaches ROC-AUC 0.99 on unseen clients, which is what a disguised label looks like, and planting it in the full model sends the score to about 1.0.
- **Overlapping windows leak too.** The 90-day CTR alone reaches about 0.87, because a third of its days *are* the label window. Adding it lifts the full model from 0.79 to about 0.89, an improvement that would be fake.
- **My own columns look plausible, not perfect.** The strongest (`ratio_prev`, `ctr_prev`) sit around 0.75 alone.
- **`trend_pct` scores 0.41**, meaning it carries an *inverted* signal. That's a reason for suspicion, not a free feature: it's computed over the window I predict, so it stays out.
- **Removing `avg_position`** barely changes the honest score (0.79 either way), so its 90-day overlap isn't what carries the result. I keep it and name it as a limitation.

## 4. What I excluded and why

Fields I refused to use as features, one line of why each:

In [4]:
excluded = pd.DataFrame([
    ("impressions/clicks/sessions_last_30d", "the label window itself"),
    ("impressions_90d, clicks_90d, sessions_90d, pageviews_90d, users_90d", "90-day totals contain the label window"),
    ("ctr, engagement_rate, scroll_rate, ai_traffic_pct", "90-day rates that contain the label window"),
    ("engaged_sessions_90d, scroll_events_90d, ai_sessions_90d", "90-day counts that contain the label window"),
    ("days_with_impressions, days_with_sessions", "counted over 90 days including the label window"),
    ("trend_pct, trend_direction, is_declining_label", "computed from the windows I predict; the dataset's own target"),
    ("days_since_last_update, freshness_tier", "export snapshot: an update inside the label window would show up"),
    ("impression_tier, position_tier, age_tier, *_count_tier", "pre-binned copies of other columns (90-day or duplicate)"),
    ("provider_used, model_used", "content-production settings, not a page signal; 72% / 19% missing"),
    ("search_volume, competition, competition_level, cpc", "keyword-level, ~8% missing; out of scope for a CTR-for-position question"),
    ("char_count", "near-duplicate of word_count, 26% missing"),
    ("client_id, content_id", "pseudonymous IDs: grouping the split only, never features (privacy + memorization)"),
], columns=["excluded field(s)", "why"])
print(excluded.to_string(index=False))
used = set(X.columns)
print(f"\n{len(used)} feature columns kept; none of them is in the excluded list:",
      not any(e.split(",")[0].strip() in used for e in excluded["excluded field(s)"]))

                                                  excluded field(s)                                                                                why
                               impressions/clicks/sessions_last_30d                                                            the label window itself
impressions_90d, clicks_90d, sessions_90d, pageviews_90d, users_90d                                             90-day totals contain the label window
                  ctr, engagement_rate, scroll_rate, ai_traffic_pct                                         90-day rates that contain the label window
           engaged_sessions_90d, scroll_events_90d, ai_sessions_90d                                        90-day counts that contain the label window
                          days_with_impressions, days_with_sessions                                    counted over 90 days including the label window
                     trend_pct, trend_direction, is_declining_label                      compu

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.